# 1. Call it

**One Python session, with a C++ compiler inside it.** `cppjit` carries Clang and LLVM. It reads a C++ header the way a compiler does, and lets Python call what it finds, with no bindings written by anyone.

The header below is the one that runs through the whole SC26 talk: one explicit-Euler step of logistic growth, templated on the number type. We include the file as it is on disk. Its checksum is printed so you can see nothing was pasted or rewritten.

In [1]:
import hashlib, time
import cppjit
print("cppjit", cppjit.__version__)
print(open("kernel/logistic_step.h").read())
print("sha256", hashlib.sha256(open("kernel/logistic_step.h", "rb").read()).hexdigest()[:16])

cppjit 0.1.0b1.dev0
// logistic_step.h -- one explicit-Euler step of logistic growth, dN/dt = r*N*(1 - N/K):
//
//     N_{t+1} = N_t + r * N_t * (1 - N_t / K)
//
// The one header the whole tutorial runs on: called from Python through cppjit, differentiated by Clad, and the
// proliferation law inside the tumour twin. Templated on the scalar type so one source serves float and double and
// so Clad differentiates the generic body, not a type-erased wrapper.
#ifndef SC26_KERNEL_LOGISTIC_STEP_H
#define SC26_KERNEL_LOGISTIC_STEP_H

template <typename T>
T logistic_step(T N, T r, T K) {
    T growth = r * N * (T(1) - N / K);
    return N + growth;
}

#endif // SC26_KERNEL_LOGISTIC_STEP_H

sha256 78e29d87fcb7b428


In [2]:
cppjit.add_include_path("kernel")
cppjit.include("logistic_step.h")            # the header, as written: parsed by Clang inside the session
N = 50.0
for _ in range(60):
    N = cppjit.gbl.logistic_step["double"](N, 0.30, 2000.0)
print(N)                                      # 1999.999842435456 in every rung of the talk
assert N == 1999.999842435456

1999.999842435456


## The template, instantiated on demand

`logistic_step` is a template. Asking for `["float"]` makes the compiler instantiate it at `float`, now, in the session. Nothing was rebuilt. The float trajectory drifts from the double one: the same law, a different number type.

In [3]:
Nd, Nf = 50.0, 50.0
for _ in range(60):
    Nd = cppjit.gbl.logistic_step["double"](Nd, 0.30, 2000.0)
    Nf = cppjit.gbl.logistic_step["float"](Nf, 0.30, 2000.0)
print(f"double {Nd!r}\nfloat  {Nf!r}\ndiff   {Nd - Nf:.3e}")

double 1999.999842435456
float  1999.9998779296875
diff   -3.549e-05


## Zero-copy: NumPy and C++ share the memory

A `std::vector<double>` made on the C++ side is viewed by NumPy without a copy. A write on either side is visible on the other. This is what lets a Python session drive a C++ model without moving its state around.

In [4]:
import numpy as np
cppjit.cppdef('''
#include <vector>
std::vector<double> trajectory(double N0, double r, double K, int steps) {
    std::vector<double> out(steps + 1); out[0] = N0;
    for (int i = 0; i < steps; ++i) out[i + 1] = logistic_step<double>(out[i], r, K);
    return out;
}''')
vec = cppjit.gbl.trajectory(50.0, 0.30, 2000.0, 60)
view = np.asarray(vec.data())                 # a NumPy view over the C++ buffer
print("owns its data:", view.flags["OWNDATA"], "| same address:", hex(view.ctypes.data) == hex(cppjit.addressof(vec.data())))
view[0] = -1.0;  assert vec[0]  == -1.0        # a NumPy write, seen from C++
vec[0]  = 50.0;  assert view[0] ==  50.0       # a C++ write, seen from NumPy
print(view[:5], "...", view[-1])

owns its data: False | same address: True
[ 50.          64.625       83.38604141 107.35886904 137.83764074] ... 1999.999842435456


## What the boundary costs, and what it buys

Crossing from Python into compiled code costs about a microsecond per call, whichever way the binding was made; for a one-line law that is more than the arithmetic itself. The point of calling C++ is to put the *loop* on the other side of the boundary: one call, a million steps.

In [5]:
def step_py(N, r, K): return N + r * N * (1 - N / K)
STEPS = 1_000_000
t0 = time.perf_counter(); n = 50.0
for _ in range(STEPS): n = step_py(n, 0.30, 2000.0)
t_py = time.perf_counter() - t0
t0 = time.perf_counter(); v = cppjit.gbl.trajectory(50.0, 0.30, 2000.0, STEPS); t_cpp = time.perf_counter() - t0
print(f"{STEPS:,} steps: pure Python loop {t_py * 1e3:7.1f} ms   |   one call into C++ {t_cpp * 1e3:7.1f} ms   ({t_py / t_cpp:.0f}x)")
t0 = time.perf_counter(); n = 50.0
for _ in range(200_000): n = cppjit.gbl.logistic_step["double"](n, 0.30, 2000.0)
print(f"and the boundary itself: {(time.perf_counter() - t0) / 200_000 * 1e6:.2f} us per call when every step is its own call")

1,000,000 steps: pure Python loop   318.5 ms   |   one call into C++    17.4 ms   (18x)


and the boundary itself: 1.29 us per call when every step is its own call


## Play with it

- Change `r` or `K` and watch the fixed point. Change the number of steps.
- Instantiate the template at `long double` (`["long double"]`), or write your own header and `cppjit.include` it.
- Everything you declared in `cppjit.cppdef` is a real C++ symbol: `cppjit.gbl.trajectory` is callable from any later cell.

Next: the compiler writes the derivative of this header, `2_transform_it.ipynb`.